# 🖥️ SWASH tutorial

This tutorial shows how to load in native Matlab files from the SWASH model and convert them to a Parcels-compatible `FieldSet`. The tutorial also shows how to run a simple particle tracking simulation using the SWASH data.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import parcels
import parcels.tutorial

In [ ]:
data_file, coord_file = parcels.tutorial.open_dataset(
    "SWASH_data/data", download_only=True
)
print(data_file, coord_file)

Unlike the other tutorials, this tutorial uses the original SWASH output files in Matlab format. If you want to use your own SWASH output files, you should use the `GRD.mat` and `ALL.mat` files in the `parcels.convert.swash_to_sgrid()` function below.

In [ ]:
ds = parcels.convert.swash_to_sgrid(
    data_file=data_file, coord_file=coord_file, total_depth=8.0
)
fieldset = parcels.FieldSet.from_sgrid_conventions(ds)
fieldset.describe()

Now, we can use this `FieldSet` to run a simulation (note it's very short because the dataset provided in this tutorial is only 20 seconds long).

In [ ]:
npart = 10  # number of particles to be released
y = np.linspace(1, 16, npart)
x = np.repeat(15, npart)

layerI = 0  # at which water depth, the particles are released
z = np.repeat(ds.depth.values[layerI], npart)
pset = parcels.ParticleSet(fieldset, x=x, y=y, z=z)

output_file = parcels.ParticleFile(
    "output-swash.parquet", outputdt=np.timedelta64(5, "s"), mode="w"
)
pset.execute(
    parcels.kernels.AdvectionRK2,
    runtime=np.timedelta64(20, "s"),
    dt=np.timedelta64(1, "s"),
    output_file=output_file,
    verbose_progress=False,
)

And then plot the results of the simulation, along with the water level at a given time step. The starting positions of the particles are also shown in white.

In [ ]:
df = parcels.read_particlefile("output-swash.parquet")

fig, ax = plt.subplots(figsize=(8, 4))
waterlevel = ds.isel(time=2).watlev.plot(cmap="magma", ax=ax)
for traj in df.partition_by("particle_id"):
    ax.plot(traj["x"][0], traj["y"][0], "wo", markersize=5)
    ax.plot(traj["x"], traj["y"], color="k")
ax.set_xlim([14, 16])
plt.show()